In [2]:
# Add project root to Python path and change working directory
import sys
import os
from pathlib import Path

# Get project root
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Change working directory to project root so config.yaml can be found
os.chdir(project_root)

print(f"Added to path: {project_root}")
print(f"Working directory: {os.getcwd()}")

Added to path: c:\Users\J'sen\Desktop\GitHub\finance-rag
Working directory: c:\Users\J'sen\Desktop\GitHub\finance-rag


In [ ]:
from src.indexing.vector_store import get_vector_store
from src.indexing.embedder import get_embedder

store = get_vector_store()
embedder = get_embedder()

c:\Users\J'sen\Desktop\GitHub\finance-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Test retrieve_by_ids
chunk_ids = ["151caa59-5e0c-497e-9540-eae1de9229d4"]
results = store.retrieve_by_ids(chunk_ids)

for r in results:
    print(f"Chunk ID: {r.chunk_id}")
    print(f"Company: {r.metadata['company']}")
    print(f"Doc Type: {r.metadata['document_type']}")
    print(f"Page: {r.metadata['page_number']}")
    print(f"Content: {r.content}...")
    print()

2026-02-13 23:07:34 - httpx - INFO - HTTP Request: POST https://c0ab454e-b357-44c0-b7ac-8eb8a938d950.us-east-1-1.aws.cloud.qdrant.io:6333/collections/sec_filings/points "HTTP/1.1 200 OK"
2026-02-13 23:07:34 - src.indexing.vector_store - INFO - Retrieved 1 points by ID
Chunk ID: 151caa59-5e0c-497e-9540-eae1de9229d4
Company: AAPL
Doc Type: 10-K
Page: 64
Content: Apple Inc.

| By:   | /s/ Kevan Parekh                               |
|-------|------------------------------------------------|
|       | Kevan Parekh                                   |
|       | Senior Vice President, Chief Financial Officer |

Power of Attorney...



In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range
# Filter for page_number == 64
page_filter = Filter(
    must=[
        FieldCondition(key="page_number", range=Range(gte=64, lte=64)),
        FieldCondition(key="company", match=MatchValue(value="AAPL")) 
    ]
)
# Use scroll to get all matching points (no vector search)
results = store.client.scroll(
    collection_name="sec_filings",
    scroll_filter=page_filter,
    limit=100,  # Max results to return
    with_payload=True,
    with_vectors=False
)

for result in results:
    print(result)
    print("\n\n")

2026-02-13 23:22:38 - httpx - INFO - HTTP Request: POST https://c0ab454e-b357-44c0-b7ac-8eb8a938d950.us-east-1-1.aws.cloud.qdrant.io:6333/collections/sec_filings/points/scroll "HTTP/1.1 200 OK"
[Record(id='151caa59-5e0c-497e-9540-eae1de9229d4', payload={'content': 'Apple Inc.\n\n| By:   | /s/ Kevan Parekh                               |\n|-------|------------------------------------------------|\n|       | Kevan Parekh                                   |\n|       | Senior Vice President, Chief Financial Officer |\n\nPower of Attorney', 'company': 'AAPL', 'document_type': '10-K', 'filing_date': '2025-09-27', 'page_number': 64, 'chunk_index': 142, 'parent_doc_id': 'aba586c4-df67-45d5-8d62-7303c73eaa80', 'has_table': True, 'has_chart': False, 'token_count': 66}, vector=None, shard_key=None, order_value=None), Record(id='3127a61e-f3ed-4d86-9e00-5f16606890f2', payload={'content': 'Pursuant to the requirements of the Securities Exchange Act of 1934, this report has been signed below by the f